In [1]:
import urllib.request

url = "https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt"
urllib.request.urlretrieve(url, "input.txt")

#!curl -o input.txt https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt


('input.txt', <http.client.HTTPMessage at 0x20a72f39a50>)

In [2]:

with open('input.txt', 'r', encoding='utf-8') as f:
    text = f.read()

print(f"Length of dataset in characters: {len(text)}")

Length of dataset in characters: 1115394


In [3]:
# Looking at some data from the dataset
print(text[:100])

First Citizen:
Before we proceed any further, hear me speak.

All:
Speak, speak.

First Citizen:
You


In [4]:
chars = sorted(list(set(text))) # Unique characters occuring in our text
vocab_size = len(chars)
print("".join(chars))
print(vocab_size)



 !$&',-.3:;?ABCDEFGHIJKLMNOPQRSTUVWXYZabcdefghijklmnopqrstuvwxyz
65


In [5]:
# create a mapping from characters to integers
stoi = {ch:i for i, ch  in enumerate(chars)}
itos = {i:ch for i, ch  in enumerate(chars)}
encode = lambda s: [stoi[c] for c in s] # encoder : take a string, output a list of string
decode = lambda l: "".join([itos[i] for i in l])# decoder: take a list of integers, output a string
print(encode("hii  there"))
print(decode(encode("hii  there"))) 

[46, 47, 47, 1, 1, 58, 46, 43, 56, 43]
hii  there


Sentence Piece(subword) and tiktoken (bpe(byte pair encode), used in gpt-2)

In [6]:
# encoding the entire text dataset and store it into a torch.tensor
import torch


In [7]:
data = torch.tensor(encode(text), dtype=torch.long)
print(data.shape, data.dtype)
print(data[:100]) # the 100 characters we looked at earlier will to 

torch.Size([1115394]) torch.int64
tensor([18, 47, 56, 57, 58,  1, 15, 47, 58, 47, 64, 43, 52, 10,  0, 14, 43, 44,
        53, 56, 43,  1, 61, 43,  1, 54, 56, 53, 41, 43, 43, 42,  1, 39, 52, 63,
         1, 44, 59, 56, 58, 46, 43, 56,  6,  1, 46, 43, 39, 56,  1, 51, 43,  1,
        57, 54, 43, 39, 49,  8,  0,  0, 13, 50, 50, 10,  0, 31, 54, 43, 39, 49,
         6,  1, 57, 54, 43, 39, 49,  8,  0,  0, 18, 47, 56, 57, 58,  1, 15, 47,
        58, 47, 64, 43, 52, 10,  0, 37, 53, 59])


In [8]:
# Let's split the dataset for training
n = int(0.9*len(data))
train_data = data[:n]
val_data = data[n:]

In [9]:
block_size = 8
train_data[:block_size+1]

tensor([18, 47, 56, 57, 58,  1, 15, 47, 58])

In [10]:
x = train_data[:block_size]
y = train_data[1:block_size+1]
for t in range(block_size):
    context = x[:t+1]
    target = y[t]
    print(f"when input is  {context} the target: {target}")

when input is  tensor([18]) the target: 47
when input is  tensor([18, 47]) the target: 56
when input is  tensor([18, 47, 56]) the target: 57
when input is  tensor([18, 47, 56, 57]) the target: 58
when input is  tensor([18, 47, 56, 57, 58]) the target: 1
when input is  tensor([18, 47, 56, 57, 58,  1]) the target: 15
when input is  tensor([18, 47, 56, 57, 58,  1, 15]) the target: 47
when input is  tensor([18, 47, 56, 57, 58,  1, 15, 47]) the target: 58


In [11]:
torch.manual_seed(1337)
batch_size = 4 # how many independent sequences will we process in //rl? (Batch Dimension)
block_size = 8 # what is the maximum context length for prediction (Timed Dimension)

def get_batch(split):
    #generate a small batch of data of inputs x and targets y
    data = train_data if split == 'train' else val_data
    ix = torch.randint(len(data) - block_size,  (batch_size,))
    x = torch.stack([data[i:i+block_size] for i in ix])
    y = torch.stack([data[i+1:i+block_size+1] for i in ix])
    return x, y

xb, yb = get_batch('train')
print("inputs:")
print(xb.shape)
print(xb)
print("targets:")
print(yb.shape)
print(yb)

print('-------')

for b in range(batch_size): # batch dimension
    for t in range(block_size): # time dimension
        context = xb[b, :t+1]
        target = yb[b, t]
        print(f"when input is {context.tolist()} the target: {target}")

inputs:
torch.Size([4, 8])
tensor([[24, 43, 58,  5, 57,  1, 46, 43],
        [44, 53, 56,  1, 58, 46, 39, 58],
        [52, 58,  1, 58, 46, 39, 58,  1],
        [25, 17, 27, 10,  0, 21,  1, 54]])
targets:
torch.Size([4, 8])
tensor([[43, 58,  5, 57,  1, 46, 43, 39],
        [53, 56,  1, 58, 46, 39, 58,  1],
        [58,  1, 58, 46, 39, 58,  1, 46],
        [17, 27, 10,  0, 21,  1, 54, 39]])
-------
when input is [24] the target: 43
when input is [24, 43] the target: 58
when input is [24, 43, 58] the target: 5
when input is [24, 43, 58, 5] the target: 57
when input is [24, 43, 58, 5, 57] the target: 1
when input is [24, 43, 58, 5, 57, 1] the target: 46
when input is [24, 43, 58, 5, 57, 1, 46] the target: 43
when input is [24, 43, 58, 5, 57, 1, 46, 43] the target: 39
when input is [44] the target: 53
when input is [44, 53] the target: 56
when input is [44, 53, 56] the target: 1
when input is [44, 53, 56, 1] the target: 58
when input is [44, 53, 56, 1, 58] the target: 46
when input is [44,

In [12]:
print(xb) # our input to the tranformer

tensor([[24, 43, 58,  5, 57,  1, 46, 43],
        [44, 53, 56,  1, 58, 46, 39, 58],
        [52, 58,  1, 58, 46, 39, 58,  1],
        [25, 17, 27, 10,  0, 21,  1, 54]])


In [13]:
import torch.nn as nn
from torch.nn import functional as F
torch.manual_seed(1337)

class BigramLanguageModel(nn.Module):
    
    def __init__(self, vocab_size):
        super().__init__()
        # each token directly reads off the logits for the next token from a lookup table
        self.token_embedding_table = nn.Embedding(vocab_size, vocab_size) # C Channel (length of the chars)

    def forward(self, idx, targets=None):

        # idx and targets are both (B, T) tensor of integers
        logits = self.token_embedding_table(idx) # (B, T, C) batch i.e. 4, Time i.e. 8 and Channel i.e. 65
        if targets is None:
            loss = None
        else:
            B, T, C = logits.shape
            logits = logits.view(B*T, C) # this is like 2D streched out
            targets = targets.view(B*T)  # we can also leave (-1) pytorch will figure it out itself
            loss = F.cross_entropy(logits, targets) # it requires B, C instead of B T C

        return logits, loss

    def generate(self, idx, max_new_tokens):
        # idx is (B,  T) array of indices in the current context
        for _ in range(max_new_tokens):
            # get the preds
            logits, loss  = self(idx)
            # focus only on the last time step
            logits = logits[:, -1, :] # becomes (B, C)
            # apply softmax to get probabilities
            probs = F.softmax(logits, dim=-1) # (B, C)
            # sample from the distribution
            idx_next = torch.multinomial(probs, num_samples=1) # (B, 1)
            # append sampled index to the running sequence
            idx = torch.cat((idx, idx_next), dim=1) # (B, T+1)
        return idx



m = BigramLanguageModel(vocab_size)
logits, loss = m(xb, yb)
print(logits.shape)
print(loss)
idx = torch.zeros((1,1) , dtype=torch.long) #(B=1. T=1)
print(decode(m.generate(idx, max_new_tokens=100)[0].tolist()))

torch.Size([32, 65])
tensor(4.8786, grad_fn=<NllLossBackward0>)

Sr?qP-QWktXoL&jLDJgOLVz'RIoDqHdhsV&vLLxatjscMpwLERSPyao.qfzs$Ys$zF-w,;eEkzxjgCKFChs!iWW.ObzDnxA Ms$3


In [14]:
# create a PyTorch optimiser
optimizer = torch.optim.AdamW(m.parameters(), lr=1e-3)

In [15]:
batch_size = 32
for steps in range(10000):

    # sample a batch of data
    xb, yb = get_batch("train")

    #evaluate the loss
    logits, loss = m(xb, yb)
    optimizer.zero_grad(set_to_none=True)
    loss.backward()
    optimizer.step()

print(loss.item())

2.5727508068084717


In [16]:
print(decode(m.generate(idx = torch.zeros((1,1) , dtype=torch.long), max_new_tokens=500)[0].tolist()))


Iyoteng h hasbe pave pirance
Rie hicomyonthar's
Plinseard ith henoure wounonthioneir thondy, y heltieiengerofo'dsssit ey
KIN d pe wither vouprrouthercc.
hathe; d!
My hind tt hinig t ouchos tes; st yo hind wotte grotonear 'so it t jod weancotha:
h hay.JUCle n prids, r loncave w hollular s O:
HIs; ht anjx?

DUThinqunt.

LaZAnde.
athave l.
KEONH:
ARThanco be y,-hedarwnoddy scace, tridesar, wnl'shenous s ls, theresseys
PlorseelapinghiybHen yof GLUCEN t l-t E:
I hisgothers je are!-e!
QLYotouciullle'z


## The Mathematical Trick for Self-Attention

but before attention understand these few concepts

In [17]:
torch.randn(1,2,3)

tensor([[[-0.8420, -1.6495,  1.2688],
         [-0.8229, -0.3710, -0.9274]]])

### 1. Random Normal Initialization (`randn`)

`randn(d0, d1, d2, ...)` creates a multi-dimensional array (tensor) filled with random numbers sampled from a **standard normal distribution** (mean = 0, standard deviation = 1).

*   **Shape Arguments:** The numbers specify the size of each dimension. For example, `randn(1, 2, 3)` outputs a tensor of shape `(1, 2, 3)`.
*   **GPT Context (B, T, C):** We often use this to initialize batch data or weights, where shape is `(Batch size, Sequence length, Embedding dimension)`.

---

### 2. Difference Between Vector, Encoding, and Embedding

| Concept | Definition | Is it Learned? | Captures Semantics? | Example |
| :--- | :--- | :---: | :---: | :--- |
| **Vector** | A 1D array of numbers representing a point in space. | N/A | No | `[1.2, -0.4, 3.1]` |
| **Encoding** | Predefined, rule-based mapping from discrete symbols to numbers. | **No** | No | ASCII (`'A'` $\to$ `65`) or One-Hot (`[1, 0, 0]`) |
| **Embedding** | Dense vector representations where values are learned during training. | **Yes** | **Yes** | PyTorch `nn.Embedding(65, 65)` |

#### Concept Flow:
`Raw Text ("cat")` $\to$ `Encoding (Token ID 4)` $\to$ `Embedding Lookup (Row 4 in Table)` $\to$ `Dense Vector ([0.25, -0.89, ...])`


In [18]:
# consider the following toy example:

torch.manual_seed(1337)
B, T, C = 4,8,2 # btch, time, channels 
x = torch.randn(B,T,C)
x.shape

torch.Size([4, 8, 2])

In [19]:
# we want x[b,t]=mean_{i<=t} x[b,i] (mean of all the previous time steps up to and including t)
xbow = torch.zeros((B,T,C)) # bow is Bag Of Words
# BOW is a way of representing text data where we count the frequency of each word in a document, ignoring the order of the words. In this case, we are creating a bag of words representation for our input data, where each element in xbow represents the mean of all previous time steps up to and including t for each batch and channel.
for b in range(B):
    for t in range(T):
        xprev = x[b,:t+1] # (t, C)
        xbow[b,t] = torch.mean(xprev, 0) 
        

In [20]:
wei = torch.tril(torch.ones(T, T))
wei = wei / torch.sum(wei, dim=1, keepdim=True)
xbow2 = wei @ x # (B, T, T) @ (B, T, C) ---->  (B, T, C)
torch.allclose(xbow, xbow2) # check if they are equal

False

In [21]:
torch.manual_seed(42)
a = torch.ones(3,3)
b = torch.randint(0,10,(3,2)).float()
c = a@b # matrix multiplication
print("a=")
print(a)
print("b=")
print(b)
print("c=")
print(c)

a=
tensor([[1., 1., 1.],
        [1., 1., 1.],
        [1., 1., 1.]])
b=
tensor([[2., 7.],
        [6., 4.],
        [6., 5.]])
c=
tensor([[14., 16.],
        [14., 16.],
        [14., 16.]])


# Bag of Words (BoW)

| Goal | Formula |
|--------|--------|
| Average all previous tokens up to position `t` | $$x_{bow}[b,t]=\frac1{t+1}\sum_{i=0}^{t}x[b,i]$$ |

---

## Input

| Tensor | Shape |
|----------|----------|
| `x` | `(B,T,C)` |
| `B` | Batch Size |
| `T` | Sequence Length |
| `C` | Embedding Dimension |

Example:

$$
X=
\begin{bmatrix}
1 & 2\\
3 & 4\\
5 & 6
\end{bmatrix}
$$

---

## Loop Implementation

| Code | Meaning |
|--------|--------|
| `x[b,:t+1]` | All tokens up to timestep `t` |
| `torch.mean(..., dim=0)` | Average across time |
| `xbow[b,t]` | Store running average |

```python
for b in range(B):
    for t in range(T):
        xprev = x[b,:t+1]
        xbow[b,t] = torch.mean(xprev, dim=0)
```

---

## Running Average Examples

| t=0 | t=1 | t=2 |
|------|------|------|
| $$[1,2]$$ | $$\frac{[1,2]+[3,4]}{2}=[2,3]$$ | $$\frac{[1,2]+[3,4]+[5,6]}{3}=[3,4]$$ |

Result:

$$
X_{bow}
=
\begin{bmatrix}
1 & 2\\
2 & 3\\
3 & 4
\end{bmatrix}
$$

---

# Matrix Multiplication Trick

Instead of recomputing means every timestep, create a matrix that already contains the averaging rules.

| Weight Matrix | Meaning |
|---------------|----------|
| $$\begin{bmatrix}1&0&0\\\frac12&\frac12&0\\\frac13&\frac13&\frac13\end{bmatrix}$$ | Row 1 → average first token<br>Row 2 → average first 2 tokens<br>Row 3 → average first 3 tokens |

---

## What Each Row Does

| Row 1 | Row 2 | Row 3 |
|--------|--------|--------|
| $$[1,0,0]$$ | $$[\frac12,\frac12,0]$$ | $$[\frac13,\frac13,\frac13]$$ |
| Picks token 1 | Averages first 2 tokens | Averages first 3 tokens |

---

## MatMul Calculations

| Row 1 | Row 2 | Row 3 |
|--------|--------|--------|
| $$[1,0,0]X=[1,2]$$ | $$[\frac12,\frac12,0]X=[2,3]$$ | $$[\frac13,\frac13,\frac13]X=[3,4]$$ |

Result:

$$
WX=
\begin{bmatrix}
1&2\\
2&3\\
3&4
\end{bmatrix}
$$

---

# Loop vs MatMul

| Loop Approach | MatMul Approach |
|---------------|----------------|
| Python executes every iteration | One tensor operation |
| Repeated slicing | No repeated slicing |
| Repeated mean calculations | All averages computed simultaneously |
| Harder to parallelize | Highly parallel |
| Slower on GPU | Extremely GPU friendly |
| Easier to understand initially | Slightly more abstract |

---

# Mental Model

| Loop Thinking | Matrix Thinking |
|---------------|----------------|
| "Compute an average for each timestep." | "Build a matrix containing all averaging rules." |
| Work done repeatedly | Work encoded once |
| Procedural | Linear Algebra |
| Python driven | GPU driven |

---

# Why Deep Learning Loves MatMul

| Traditional View | Hardware View |
|------------------|---------------|
| We are averaging tokens | We are performing matrix multiplication |
| We are computing attention | We are performing matrix multiplication |
| We are running a linear layer | We are performing matrix multiplication |

Modern GPUs are optimized primarily for:

$$
A @ B
$$

which is why deep learning frameworks try to express everything as matrix multiplications.

---

# Key Takeaway

| Loop Version | Matrix Version |
|-------------|---------------|
| `mean(x[b,:t+1])` | `W @ X` |
| Easier to read | Faster |
| Good for learning | Good for training |
| Same math | Same math |

Both produce exactly the same output.

In [22]:
torch.manual_seed(42)
a = torch.tril(torch.ones(3,3))
a = a / torch.sum(a, dim=1, keepdim=True) # normalize the rows
b = torch.randint(0,10,(3,2)).float()
c = a@b # matrix multiplication
print("a=")
print(a)
print("b=")
print(b)
print("c=")
print(c)

a=
tensor([[1.0000, 0.0000, 0.0000],
        [0.5000, 0.5000, 0.0000],
        [0.3333, 0.3333, 0.3333]])
b=
tensor([[2., 7.],
        [6., 4.],
        [6., 5.]])
c=
tensor([[2.0000, 7.0000],
        [4.0000, 5.5000],
        [4.6667, 5.3333]])


```python
a = a / torch.sum(a, dim=1, keepdim=True)
```

| Part | Meaning |
|--------|--------|
| `dim=1` | Sum each row |
| `keepdim=True` | Keep result as `(N,1)` |
| Broadcasting | Expand `(N,1)` → `(N,M)` |
| Division | Normalize each row |

Example:

```python
[[1,1,1],
 [1,1,0],
 [1,0,0]]
```

↓

```python
[[1/3,1/3,1/3],
 [1/2,1/2,0],
 [1,0,0]]
```

Every row sums to `1`.
```

In [23]:
# Version 2 : Use Matrix Multiplication
wei = torch.tril(torch.ones(T, T))
wei = wei / torch.sum(wei, dim=1, keepdim=True)
xbow2 = wei @ x # (B, T, T) @ (B, T, C) ---->  (B, T, C)
torch.allclose(xbow, xbow2) # check if they are equal

False

In [24]:
print(x.shape)      # should be (B,T,C)
print(wei.shape)    # should be (T,T)
print(torch.max(torch.abs(xbow - xbow2)))
print(xbow.dtype)
print(xbow2.dtype)
print(torch.allclose(xbow, xbow2))
print(torch.allclose(xbow, xbow2, atol=1e-7))

torch.Size([4, 8, 2])
torch.Size([8, 8])
tensor(3.2363e-08)
torch.float32
torch.float32
False
True


# Floating Point Error & `torch.allclose()`

## Key Idea

Same math ≠ same bits.

Different computation paths:

```python
# Loop
torch.mean(...)

# Matrix multiply
wei @ x
```

can produce tiny floating-point differences.

---

## Formula

`torch.allclose(a, b)` checks:

$$
|a-b| \leq atol + rtol \cdot |b|
$$

Default values:

| Parameter | Value |
|-----------|--------|
| `atol` | `1e-8` | (Absolute Tolerance: Fixed allowable error)
| `rtol` | `1e-5` | (Relative Tolerance: Error allowed relative to value size)

---

## Example

```python
torch.max(torch.abs(xbow - xbow2))
```

Output:

```python
tensor(3.2363e-08)
```

Comparison:

```python
torch.equal(xbow, xbow2)
```

```python
False
```

```python
torch.allclose(xbow, xbow2)
```

```python
False
```

```python
torch.allclose(xbow, xbow2, atol=1e-7)
```

```python
True
```

---

## Equality Comparison

| Function | Meaning |
|----------|----------|
| `torch.equal(a,b)` | Exact bit-for-bit equality |
| `torch.allclose(a,b)` | Numerically close |
| `torch.isclose(a,b)` | Element-wise closeness |
| `torch.max(torch.abs(a-b))` | Maximum error |

---

## Common Confusions

| Confusion | Reality |
|------------|----------|
| `allclose=False` ⇒ code is wrong | May be floating-point tolerance |
| `equal=False` ⇒ tensors are very different | Difference may be `1e-8` |
| Same formula ⇒ same output bits | Operation order matters |
| Loop and matmul should match exactly | Usually only numerically |

---

## Debug Checklist

```python
torch.max(torch.abs(a-b))
```

```python
torch.equal(a,b)
```

```python
torch.allclose(a,b)
```

```python
torch.allclose(a,b, atol=1e-7)
```

Locate mismatches:

```python
torch.nonzero(~torch.isclose(a,b))
```

---

## Mental Model

Floating-point numbers are approximations.

Tiny errors accumulate differently depending on the computation path.

---

## Quick Reference

| Concept | Remember |
|----------|------------|
| `torch.equal()` | Exact bits |
| `torch.allclose()` | Numerical equality |
| Default `atol` | `1e-8` |
| Default `rtol` | `1e-5` |
| Error `1e-8 ~ 1e-7` | Usually harmless |
| Loop vs MatMul | Same math, different rounding |
| First debug step | `torch.max(torch.abs(a-b))` |
| Your case | `3.2363e-08` |
| `allclose=False`, `atol=1e-7=True` | Tolerance issue, not logic issue |

In [25]:
# Version 3 : Use Softmax 
tril = torch.tril(torch.ones(T, T))
wei = torch.zeros((T, T))
wei = wei.masked_fill(tril == 0, float('-inf')) # fill the upper triangular part with -inf
wei = F.softmax(wei, dim=-1) # apply softmax to get probabilities
xbow3 = wei @ x # (B, T, T) @ (B, T, C) ---->  (B, T, C)
print(torch.allclose(xbow2, xbow3))
print(torch.allclose(xbow, xbow3, atol=1e-7)) # not all close because of the floating point precision issues, but they are very close.

True
True


# `masked_fill()`

## Syntax

```python
tensor.masked_fill(mask, value)
```

Replace elements where:

```python
mask == True
```

with:

```python
value
```

---

## Basic Example

```python
x = tensor([
 [1,2],
 [3,4]
])

mask = tensor([
 [True, False],
 [False, True]
])
```

```python
x.masked_fill(mask, 0)
```

Result:

```python
[
 [0,2],
 [3,0]
]
```

---

## Rule

| Mask Value | Action |
|------------|---------|
| `True` | Replace |
| `False` | Keep original |

---

## Common Uses

| Use Case | Example |
|-----------|-----------|
| Zero out values | `x.masked_fill(mask, 0)` |
| Ignore positions | `x.masked_fill(mask, -1)` |
| Attention masking | `x.masked_fill(mask, -inf)` |
| Pad masking | `x.masked_fill(pad_mask, 0)` |

---

## Attention Example

```python
wei =
[
 [1,2,3],
 [4,5,6],
 [7,8,9]
]
```

Future-token mask:

```python
mask =
[
 [False, True,  True ],
 [False, False, True ],
 [False, False, False]
]
```

```python
wei = wei.masked_fill(mask, -inf)
```

Result:

```python
[
 [1,-inf,-inf],
 [4,5,-inf],
 [7,8,9]
]
```

---

## Why `-inf`?

After:

```python
softmax(wei)
```

```python
e^(-inf) = 0
```

so those positions get probability:

```python
0
```

and cannot be attended to.

---

## Shape Rule

```python
tensor.shape == mask.shape
```

or

```python
mask
```

must be broadcastable to the tensor shape.

---

## Mental Model

```text
masked_fill =
"replace selected positions"
```

Mask decides **where**.

Value decides **what to replace with**.

In [26]:
tril==0 , tril==1  #  0 is for upper triangular part and 1 is for lower triangular part in case of masking

(tensor([[False,  True,  True,  True,  True,  True,  True,  True],
         [False, False,  True,  True,  True,  True,  True,  True],
         [False, False, False,  True,  True,  True,  True,  True],
         [False, False, False, False,  True,  True,  True,  True],
         [False, False, False, False, False,  True,  True,  True],
         [False, False, False, False, False, False,  True,  True],
         [False, False, False, False, False, False, False,  True],
         [False, False, False, False, False, False, False, False]]),
 tensor([[ True, False, False, False, False, False, False, False],
         [ True,  True, False, False, False, False, False, False],
         [ True,  True,  True, False, False, False, False, False],
         [ True,  True,  True,  True, False, False, False, False],
         [ True,  True,  True,  True,  True, False, False, False],
         [ True,  True,  True,  True,  True,  True, False, False],
         [ True,  True,  True,  True,  True,  True,  True, F

# `F.softmax()`

## Syntax

```python
wei = F.softmax(wei, dim=-1)
```

---

## Formula

For a row:

$$
\text{softmax}(x_i)
=
\frac{e^{x_i}}
{\sum_j e^{x_j}}
$$

Steps:

```text
1. Exponentiate
2. Sum
3. Divide by sum
```

Result:

```text
All values ∈ (0,1)
Row sums to 1
```

---

## Example

Input:

```python
[1,2,3]
```

### Step 1: Exponentiate

```python
[e¹,e²,e³]
```

≈

```python
[2.7, 7.4, 20.1]
```

### Step 2: Sum

```python
2.7 + 7.4 + 20.1
=
30.2
```

### Step 3: Normalize

```python
[
 2.7/30.2,
 7.4/30.2,
 20.1/30.2
]
```

≈

```python
[
 0.09,
 0.24,
 0.67
]
```

---

## Why Not Just Divide By The Sum?

### Simple Normalization

```python
[1,2,3]
```

↓

```python
[
 1/6,
 2/6,
 3/6
]
```

↓

```python
[
 0.17,
 0.33,
 0.50
]
```

### Softmax

```python
[1,2,3]
```

↓

```python
[
 0.09,
 0.24,
 0.67
]
```

Softmax exaggerates differences.

---

## Attention Example

Scores:

```python
[2, 8, 4]
```

Normalize directly:

```python
[
 0.14,
 0.57,
 0.29
]
```

Softmax:

```python
[
 0.002,
 0.98,
 0.018
]
```

The most important token gets most of the attention.

---

## Why Use `e^x`?

| Property | Why Useful |
|-----------|-----------|
| Always positive | No negative probabilities |
| Larger x grows much faster | Highlights important tokens |
| Differentiable | Works with gradient descent |
| Sum can be normalized | Produces probabilities |

---

## `-inf` Trick

Input:

```python
[
 2,
 5,
 -inf
]
```

Exponentiate:

```python
[
 e²,
 e⁵,
 e^-inf
]
```

↓

```python
[
 7.4,
 148,
 0
]
```

Normalize:

```python
[
 7.4/155.4,
 148/155.4,
 0/155.4
]
```

↓

```python
[
 0.048,
 0.952,
 0
]
```

Masked position gets exactly:

```python
0
```

---

## In Attention

Before Softmax:

```python
Q @ K.T
```

contains:

```text
similarity scores
```

Example:

```python
[
 1.2,
 3.7,
 0.5
]
```

After Softmax:

```python
[
 0.07,
 0.88,
 0.05
]
```

Now they become:

```text
attention weights
```

that sum to 1.

---

## Mental Model

| Before Softmax | After Softmax |
|---------------|---------------|
| Scores | Probabilities |
| Any real number | Between 0 and 1 |
| Doesn't sum to 1 | Sums to 1 |
| Similarity strength | Attention weight |

---

## Quick Reference

```python
F.softmax(x, dim=-1)
```

```text
Exponentiate
↓
Make positive
↓
Normalize
↓
Get probabilities
```

Softmax = "turn scores into attention weights".

## Mask + Softmax

```python
[0,-inf,-inf]
```

↓

```python
[e⁰,0,0]
```

↓

```python
[1,0,0]
```

because

```python
1/(1+0+0)=1
```

### Rule

```python
e^(-inf)=0
```

Masked positions get probability `0`.

Remaining positions share all probability mass.

Rows still sum to `1`.

# `torch.tril()`

## Syntax

```python
torch.tril(x)
```

Keeps the **lower triangle** and sets everything above the diagonal to `0`.

---

## Example

Input:

```python
[
 [1,2,3],
 [4,5,6],
 [7,8,9]
]
```

```python
torch.tril(x)
```

Output:

```python
[
 [1,0,0],
 [4,5,0],
 [7,8,9]
]
```

---

## Visual

```text
✓ Keep
✗ Remove
```

```text
[
 [✓ ✗ ✗]
 [✓ ✓ ✗]
 [✓ ✓ ✓]
]
```

---

## Transformer Usage

```python
torch.tril(torch.ones(T,T))
```

Example (`T=4`):

```python
[
 [1,0,0,0],
 [1,1,0,0],
 [1,1,1,0],
 [1,1,1,1]
]
```

Shape:

```python
(T,T)
```

---

## Create Causal Mask

```python
tril = torch.tril(torch.ones(T,T))
```

```python
[
 [1,0,0,0],
 [1,1,0,0],
 [1,1,1,0],
 [1,1,1,1]
]
```

Then:

```python
wei = wei.masked_fill(tril == 0, -inf)
```

Result:

```python
[
 [a,-inf,-inf,-inf],
 [a,b,-inf,-inf],
 [a,b,c,-inf],
 [a,b,c,d]
]
```

Future tokens are masked.

---

## Why?

| Row | Can See |
|------|---------|
| 0 | Token 0 |
| 1 | Tokens 0,1 |
| 2 | Tokens 0,1,2 |
| 3 | Tokens 0,1,2,3 |

Cannot look into the future.

---

## Quick Reference

| Function | Meaning |
|-----------|---------|
| `torch.tril(x)` | Lower triangle |
| `torch.triu(x)` | Upper triangle |
| `tril == 0` | Future positions |
| `masked_fill(...,-inf)` | Block future attention |

### Mental Model

```text
tril =
"tokens can only see themselves and the past"
```

In [36]:
# Version 4: Self Attention
torch.manual_seed(1337)
B,T,C = 4,8,32 # batch, time, channels
x  = torch.randn(B, T, C)

# let's see a single Head perform self-attention
head_size = 16 # hyperparameter
key = nn.Linear(C, head_size, bias = False)
query = nn.Linear(C, head_size, bias = False)
k = key(x)      # (B, T, 16)
q = query(x)    # (B, T, 16)
wei = q @ k.transpose(-2, -1)   # (B, T, 16) @ (B, 16, T) --> (B, T, T)


tril = torch.tril(torch.ones(T, T))
#wei = torch.zeros((T, T))
wei = wei.masked_fill(tril == 0, float('-inf'))
wei = F.softmax(wei, dim=-1)
out = wei @ x

out.shape

torch.Size([4, 8, 32])

In [37]:
tril

tensor([[1., 0., 0., 0., 0., 0., 0., 0.],
        [1., 1., 0., 0., 0., 0., 0., 0.],
        [1., 1., 1., 0., 0., 0., 0., 0.],
        [1., 1., 1., 1., 0., 0., 0., 0.],
        [1., 1., 1., 1., 1., 0., 0., 0.],
        [1., 1., 1., 1., 1., 1., 0., 0.],
        [1., 1., 1., 1., 1., 1., 1., 0.],
        [1., 1., 1., 1., 1., 1., 1., 1.]])

In [39]:
wei[0]

tensor([[1.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.1574, 0.8426, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.2088, 0.1646, 0.6266, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.5792, 0.1187, 0.1889, 0.1131, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.0294, 0.1052, 0.0469, 0.0276, 0.7909, 0.0000, 0.0000, 0.0000],
        [0.0176, 0.2689, 0.0215, 0.0089, 0.6812, 0.0019, 0.0000, 0.0000],
        [0.1691, 0.4066, 0.0438, 0.0416, 0.1048, 0.2012, 0.0329, 0.0000],
        [0.0210, 0.0843, 0.0555, 0.2297, 0.0573, 0.0709, 0.2423, 0.2391]],
       grad_fn=<SelectBackward0>)

In [52]:
k,q  = nn.Linear(3,2),nn.Linear(3,2)
k.weight, q.weight

(Parameter containing:
 tensor([[-0.4590,  0.3002,  0.3505],
         [ 0.3063,  0.4283,  0.2576]], requires_grad=True),
 Parameter containing:
 tensor([[ 0.3920, -0.5086, -0.0791],
         [-0.0664, -0.2746,  0.1502]], requires_grad=True))